<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-0/08-mini-proyecto-reporte-mensual/ejercicio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Ejercicio 0.8 · Miniproyecto: automatiza tu reporte mensual

**Serie:** Del Excel al Machine Learning · **Temporada 0** · 🎬 Video próximamente

Antes de empezar, guarda tu copia: **Archivo → Guardar una copia en Drive**.

Escribe tu respuesta en cada celda de código y ejecútala con **Shift + Enter**.

### Prepara los datos

La gerente de Arequipa quiere su propio reporte, y las otras sedes también. Lo vas a construir por partes.

Ejecuta estas tres celdas. Traen la limpieza de la clase y las funciones `filtrar_mes` y `resumir`. Además, `ventas` ya incluye `producto`, `categoria`, `vendedor` y `tipo` de cliente.

Algunos ejercicios usan lo que creaste en los anteriores, así que **ejecútalos en orden**.

In [ ]:
import pandas as pd

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [ ]:
def limpiar_monto(columna):
    return columna.str.replace("S/ ", "").str.replace(",", "").astype(float)


def limpiar_ventas(archivo):
    ventas = pd.read_csv(archivo, sep=";")

    # Nombres de columna
    ventas.columns = ventas.columns.str.lower().str.replace(" ", "_")
    ventas = ventas.rename(
        columns={
            "nro_venta": "id_venta",
            "cliente": "id_cliente",
            "producto": "id_producto",
            "vendedor": "id_vendedor",
            "método_de_pago": "metodo_pago",
        }
    )

    # Duplicados
    ventas = ventas.drop_duplicates()

    # Textos
    ventas["ciudad"] = ventas["ciudad"].str.strip().str.capitalize()
    ventas["metodo_pago"] = ventas["metodo_pago"].str.strip().str.capitalize()
    ventas["metodo_pago"] = ventas["metodo_pago"].replace(
        {"Yape": "Billetera digital", "Plin": "Billetera digital"}
    )
    ventas["metodo_pago"] = ventas["metodo_pago"].fillna("Sin dato")

    # Números guardados como texto
    ventas["precio_unitario"] = limpiar_monto(ventas["precio_unitario"])
    ventas["total"] = limpiar_monto(ventas["total"])
    ventas["descuento"] = ventas["descuento"].str.replace("%", "").astype(float) / 100

    # Fechas
    texto = ventas["fecha"]
    ventas["fecha"] = pd.to_datetime(texto, format="%d/%m/%Y", errors="coerce")
    ventas["fecha"] = ventas["fecha"].fillna(
        pd.to_datetime(texto, format="%Y-%m-%d", errors="coerce")
    )
    ventas["anio"] = ventas["fecha"].dt.year
    ventas["mes"] = ventas["fecha"].dt.month

    # Cantidades vacías o con errores de tipeo
    cantidad = ventas["total"] / (ventas["precio_unitario"] * (1 - ventas["descuento"]))
    ventas["cantidad"] = cantidad.round().astype(int)

    return ventas

In [ ]:
def filtrar_mes(ventas, anio, mes):
    return ventas[(ventas["anio"] == anio) & (ventas["mes"] == mes)]


def resumir(tabla, columna):
    resumen = tabla.groupby(columna)["total"].sum().sort_values(ascending=False).reset_index()
    resumen["porcentaje"] = (resumen["total"] / resumen["total"].sum() * 100).round(1)
    resumen["total"] = resumen["total"].round(2)
    return resumen


ventas = limpiar_ventas(DATOS + "ventas_crudo.csv")

productos = pd.read_csv(DATOS + "productos.csv")
empleados = pd.read_csv(DATOS + "empleados.csv")
clientes = pd.read_csv(DATOS + "clientes.csv")

productos = productos[["id_producto", "nombre", "categoria"]].rename(columns={"nombre": "producto"})
empleados = empleados[["id_empleado", "nombre"]].rename(columns={"nombre": "vendedor"})

ventas = pd.merge(ventas, productos, on="id_producto", how="left")
ventas = pd.merge(ventas, empleados, left_on="id_vendedor", right_on="id_empleado", how="left")
ventas = pd.merge(ventas, clientes[["id_cliente", "tipo"]], on="id_cliente", how="left")

print(len(ventas))
print(round(ventas["total"].sum(), 2))

### 1. Julio en números

Con `filtrar_mes`, guarda las ventas de julio de 2025 en `julio`. Muestra la suma de `total`, redondeada a 2 decimales, y cuántas ventas hubo. La suma debe dar `690485.09` y las ventas, `2513`.

In [ ]:
# Escribe tu código aquí

### 2. Contra el año pasado

Calcula la variación porcentual entre julio de 2025 y julio de 2024, redondeada a 1 decimal. La fórmula es `(este año / año pasado - 1) * 100`.

Debe dar `-1.3`. ¿Qué significa el signo?

In [ ]:
# Escribe tu código aquí

### 3. Las tablas del reporte

Con `resumir`, muestra las ventas de `julio` por `tipo` de cliente y por `metodo_pago`. Después, los **5 productos** con más soles vendidos: el primero debe ser `Shampoo 400 ml`, con `63540.0`.

In [ ]:
# Escribe tu código aquí

### 4. Una función para la sede

Crea la función `filtrar_sede(ventas, ciudad, anio, mes)`, que devuelva las ventas de esa ciudad en ese mes. Usa `filtrar_mes` dentro de tu función.

Pruébala con Arequipa, julio de 2025: la suma de `total`, redondeada a 2 decimales, debe dar `97746.82`.

In [ ]:
# Escribe tu código aquí

### 5. El reporte de la sede

Crea la función `crear_reporte_sede(ventas, ciudad, anio, mes)`, que:

1. Si no hay ventas, muestre un aviso y termine con `return`.
2. Guarde un Excel llamado, por ejemplo, `reporte_Arequipa_2025_07.xlsx`.
3. Tenga tres hojas: `Categorías`, `Vendedores` y `Top 10 productos`.
4. Tenga las columnas A y B lo bastante anchas para leer los nombres.

Pruébala con Arequipa, julio de 2025. Abre el archivo: `Vendedores` debe tener 2 filas (`57304.0` y `40442.82`). Pruébala también con una ciudad mal escrita, como `arequipa`, y comprueba que aparezca el aviso.

> 💡 Copia `crear_reporte` de la clase y quita el gráfico, `add_image` y las hojas `Resumen` y `Ciudades`. Cambia `del_mes` por las ventas de la sede en todas las tablas.

In [ ]:
# Escribe tu código aquí

### 6. Todas las sedes

Con un `for`, crea el reporte de julio de 2025 de las 6 sedes. Escribe la lista de ciudades, o pídesela a pandas con `ventas["ciudad"].unique()`.

Deben aparecer 6 archivos en la carpeta 📁.

In [ ]:
# Escribe tu código aquí

### 7. ¿Cuadra todo?

Abre la hoja `Categorías` de cada uno de los 6 archivos y suma sus columnas `total` en un **acumulador**, como en el episodio 0.3. Muestra el resultado redondeado a 2 decimales: debe coincidir con el total del ejercicio 1.

Si cuadra, tu reporte está listo para enviarse.

In [ ]:
# Escribe tu código aquí

## 🏁 ¡Terminaste la Temporada 0!

Ahora piensa en **un reporte que hagas todos los meses**. ¿Qué pasos se repiten? Empieza por automatizar uno solo, con una copia de tus datos y sin compartir información de tu empresa.

Compara tus respuestas con [la solución](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-0/08-mini-proyecto-reporte-mensual/solucion.ipynb).

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0